Madhulata Mukkamala - Nassau Candy_Factory Reallocation & Shipping Optimization Recommendation System

In [1]:
"""
PHASE 1 — Data Preparation & Encoding
Nassau Candy Factory Reallocation & Shipping Optimization Recommendation System
 
Per the project brief:
- Normalize numerical features
- Encode categorical variables (region, ship mode)
- Remove extreme outliers
- Create training-ready feature matrix
 
Run this first. It reads the raw supplied dataset and writes a cleaned,
feature-engineered version that every later phase reads from 
Usage:
 
"""

'\nPHASE 1 — Data Preparation & Encoding\nNassau Candy Factory Reallocation & Shipping Optimization Recommendation System\n\nPer the project brief:\n- Normalize numerical features\n- Encode categorical variables (region, ship mode)\n- Remove extreme outliers\n- Create training-ready feature matrix\n\nRun this first. It reads the raw supplied dataset and writes a cleaned,\nfeature-engineered version that every later phase reads from \nUsage:\n\n'

In [ ]:
# ============================================
# STEP 1.1: IMPORT REQUIRED LIBRARIES
# ============================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from datetime import datetime

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Reproducibility
RANDOM_STATE = 42

print("Libraries imported successfully!")

In [ ]:
# 1.2 Set the dataset path
file_path = "nassau-candy-factory-optimization/data/Nassau Candy Distributor.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")

In [ ]:
# 1.3 Check number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# 1.4 Display the first 5 records
df.head()

In [ ]:
#1.5 Check column names
print("Columns in dataset:\n")

for i, column in enumerate(df.columns, start=1):
    print(i, ":", column)

In [ ]:
# 1.6 Check data types
df.info()

In [ ]:
#1.7 Check missing values
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    )
})

missing_summary = missing_summary.sort_values(
    by="Missing Count",
    ascending=False
)

display(missing_summary)

total_missing = int(df.isnull().sum().sum())

print("\nTotal missing cells:", total_missing)

In [ ]:
# ============================================
# STEP 1.8: DUPLICATE ANALYSIS
# ============================================

# Exact duplicate rows
exact_duplicates = df.duplicated().sum()

# Duplicate Row IDs
duplicate_row_ids = df["Row ID"].duplicated().sum()

# Duplicate Order IDs
duplicate_order_ids = df["Order ID"].duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate Row IDs:", duplicate_row_ids)
print("Duplicate Order IDs:", duplicate_order_ids)

In [ ]:
# 1.9 Check unique values
print("Unique Products:", df["Product Name"].nunique())
print("Unique Product IDs:", df["Product ID"].nunique())
print("Unique Orders:", df["Order ID"].nunique())
print("Unique Customers:", df["Customer ID"].nunique())
print("Unique Regions:", df["Region"].nunique())
print("Unique Ship Modes:", df["Ship Mode"].nunique())
print("Unique States:", df["State/Province"].nunique())
print("Unique Countries:", df["Country/Region"].nunique())

In [ ]:
# 1.10 Display products

products = (
    df["Product Name"]
    .value_counts()
    .reset_index()
)

products.columns = ["Product Name", "Record Count"]

products

In [ ]:
#1.11 Display regions

region_counts = (
    df["Region"]
    .value_counts()
    .reset_index()
)

region_counts.columns = ["Region", "Record Count"]

region_counts

In [ ]:
#1.12 Display ship modes

ship_mode_counts = (
    df["Ship Mode"]
    .value_counts()
    .reset_index()
)

ship_mode_counts.columns = ["Ship Mode", "Record Count"]

ship_mode_counts

In [ ]:
# ============================================
# STEP 1.13: DATE CONVERSION
# ============================================

date_columns = ["Order Date", "Ship Date"]

for column in date_columns:

    df[column] = pd.to_datetime(
        df[column],
        format="%d-%m-%Y",
        errors="coerce"
    )

print("Date conversion completed.")

print("\nOrder Date range:")
print("Minimum:", df["Order Date"].min())
print("Maximum:", df["Order Date"].max())

print("\nShip Date range:")
print("Minimum:", df["Ship Date"].min())
print("Maximum:", df["Ship Date"].max())

print("\nInvalid Order Dates:", df["Order Date"].isna().sum())
print("Invalid Ship Dates:", df["Ship Date"].isna().sum())

In [ ]:
# ============================================
# STEP 1.14: CALCULATE SHIPPING LEAD TIME
# ============================================
# Ensure date columns are converted to datetime first
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d-%m-%Y", errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%d-%m-%Y", errors="coerce")

# Calculate lead time in days
df["Shipping Lead Time Days"] = (df["Ship Date"] - df["Order Date"]).dt.total_seconds() / (24 * 60 * 60)

print("Shipping lead time calculated.")
display(df[["Order ID", "Order Date", "Ship Date", "Shipping Lead Time Days"]].head(10))

In [ ]:
# ============================================
# STEP 1.15: SHIPPING LEAD TIME QUALITY CHECK
# ============================================

lead_time = df["Shipping Lead Time Days"]

print("Shipping Lead Time Summary")
display(lead_time.describe())

print(
    "Negative lead times:",
    int((lead_time < 0).sum())
)

print(
    "Zero-day lead times:",
    int((lead_time == 0).sum())
)

print(
    "Lead times exceeding 30 days:",
    int((lead_time > 30).sum())
)

print(
    "Lead times exceeding 365 days:",
    int((lead_time > 365).sum())
)

In [ ]:
# ============================================
# STEP 1.16: NUMERICAL DATA VALIDATION
# ============================================

numeric_columns = [
    "Sales",
    "Units",
    "Gross Profit",
    "Cost"
]

print("Numerical summary:")
display(df[numeric_columns].describe())

print("\nNegative values:")
display(
    (df[numeric_columns] < 0).sum().to_frame(
        name="Negative Value Count"
    )
)

print("\nZero values:")
display(
    (df[numeric_columns] == 0).sum().to_frame(
        name="Zero Value Count"
    )
)

print("\nMissing numerical values:")
display(
    df[numeric_columns].isnull().sum().to_frame(
        name="Missing Value Count"
    )
)

In [ ]:
# ============================================
# STEP 1.17: FINANCIAL RECONCILIATION CHECK
# ============================================

df["Financial Reconciliation Difference"] = (
    df["Sales"] - df["Cost"] - df["Gross Profit"]
)

print("Financial reconciliation difference:")
display(
    df["Financial Reconciliation Difference"].describe()
)

tolerance = 0.02

reconciliation_failures = (
    df["Financial Reconciliation Difference"].abs()
    > tolerance
)

print(
    "Records outside the reconciliation tolerance:",
    int(reconciliation_failures.sum())
)

In [ ]:
# ============================================
# STEP 1.18: CATEGORICAL DATA AUDIT
# ============================================

categorical_columns = [
    "Product Name",
    "Product ID",
    "Division",
    "Region",
    "Ship Mode",
    "Country/Region"
]

for column in categorical_columns:

    print(f"\nUnique values in {column}:")
    print(df[column].nunique())

    print(df[column].value_counts(dropna=False).head(20))

In [ ]:
# ============================================
# STEP 1.19: FACTORY ASSIGNMENT CHECK
# ============================================

possible_factory_columns = [
    column
    for column in df.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "factory",
            "plant",
            "warehouse",
            "origin",
            "fulfillment center"
        ]
    )
]

print("Possible factory assignment columns:")
print(possible_factory_columns)

if not possible_factory_columns:
    print(
        "\nNo explicit factory-assignment column was found."
    )
    print(
        "We need a validated product-to-factory mapping "
        "before building the factory reallocation engine."
    )

In [ ]:
# ============================================
# STEP 1.20: CREATE DATA AUDIT REPORT
# ============================================

audit_results = {
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Exact Duplicate Rows",
        "Duplicate Row IDs",
        "Duplicate Order IDs",
        "Total Missing Cells",
        "Unique Products",
        "Unique Product IDs",
        "Unique Regions",
        "Unique Shipping Modes",
        "Invalid Order Dates",
        "Invalid Ship Dates",
        "Negative Shipping Lead Times",
        "Zero-Day Shipping Lead Times",
        "Lead Times Over 365 Days",
        "Factory Assignment Columns Found"
    ],

    "Value": [
        len(df),
        len(df.columns),
        int(df.duplicated().sum()),
        int(df["Row ID"].duplicated().sum()),
        int(df["Order ID"].duplicated().sum()),
        int(df.isnull().sum().sum()),
        int(df["Product Name"].nunique()),
        int(df["Product ID"].nunique()),
        int(df["Region"].nunique()),
        int(df["Ship Mode"].nunique()),
        int(df["Order Date"].isna().sum()),
        int(df["Ship Date"].isna().sum()),
        int((df["Shipping Lead Time Days"] < 0).sum()),
        int((df["Shipping Lead Time Days"] == 0).sum()),
        int((df["Shipping Lead Time Days"] > 365).sum()),
        len(possible_factory_columns)
    ]
}

audit_df = pd.DataFrame(audit_results)

display(audit_df)

# Save the report
audit_df.to_csv(
    "nassau-candy-factory-optimization/data/nassau_candy_data_audit.csv",
    index=False
)

print("\nAudit report saved as nassau_candy_data_audit.csv")

In [ ]:
# Set the output location
output_file = Path(
    "nassau-candy-factory-optimization/data/Nassau_Candy_Cleaned_Step.csv"
)

# Create the folder if it does not exist
output_file.parent.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset, including the new column
df.to_csv(output_file, index=False)

print("Step 1 completed successfully.")
print("Saved file:", output_file)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shipping lead-time column saved:",
      "Shipping Lead Time Days" in df.columns)

In [ ]:
df_check = pd.read_csv(output_file)

assert "Shipping Lead Time Days" in df_check.columns, (
    "Shipping Lead Time Days was not saved."
)

assert len(df_check) == len(df), (
    "The saved CSV row count does not match df."
)

print("Verification successful.")
print("Saved dataset shape:", df_check.shape)
print("Columns:", df_check.columns.tolist())